In [5]:
#  Kutubxonalarni chaqiramiz
import sqlite3  # Ma'lumotlarni saqlash uchun (SQLite bazasi)
import requests  # Veb-sahifadan ma'lumot olish uchun (HTTP so'rov)
from bs4 import BeautifulSoup  # HTML kodni tahlil qilish uchun


# 1️.QADAM: Ma'lumotlar bazasini yaratish yoki unga ulanish
def setup_database():
    # "jobs.db" nomli SQLite fayliga ulanadi (agar bo‘lmasa, yangi yaratiladi)
    conn = sqlite3.connect("jobs.db")

    # Ma'lumotlar bilan ishlash uchun cursor (ko‘rsatkich) hosil qilinadi
    cursor = conn.cursor()

    # Agar "jobs" jadvali mavjud bo‘lmasa — uni yaratadi
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS jobs (
            id INTEGER,
            job_title TEXT(500),
            creator TEXT(500),
            address TEXT(500),
            date TEXT
        )
        """
    )

    # O‘zgarishlarni bazaga saqlab qo‘yadi
    conn.commit()

    # Ulanish va cursorni qaytaradi
    return conn, cursor


# 2️. QADAM: Saytdan ish e'lonlarini olish (scraping)
def scrape_jobs():
    # Ma'lumot olinadigan veb-sayt manzili
    url = "https://realpython.github.io/fake-jobs/"

    # Saytga so‘rov yuboramiz
    response = requests.get(url)

    # Sahifani BeautifulSoup yordamida HTML ko‘rinishida tahlil qilamiz
    soup = BeautifulSoup(response.text, "html.parser")

    # Bo‘sh ro‘yxat hosil qilamiz — hamma e'lonlar shu yerga joylanadi
    jobs = []

    # Har bir ish e'loniga unikal ID berish uchun boshlang‘ich qiymat
    job_id = 1

    # Sahifadagi barcha ish e'lonlarini topamiz
    for job in soup.find_all("div", class_="card-content"):

        # Har bir ishning sarlavhasini olish
        job_title = job.find("h2", class_="title").text.strip()

        # Ishni e'lon qilgan kompaniya nomini olish
        creator = job.find("h3", class_="company").text.strip()

        # Ish joyining manzilini olish
        address = job.find("p", class_="location").text.strip()

        # E'lon sanasini olish
        date = job.find("time")["datetime"]

        # Hamma ma'lumotlarni bitta tuple ko‘rinishida ro‘yxatga qo‘shamiz
        jobs.append((job_id, job_title, creator, address, date))

        # Keyingi ish uchun ID ni oshiramiz
        job_id += 1

    # Hamma e'lonlar ro‘yxatini qaytaramiz
    return jobs


# 3️. QADAM: Olingan ish e'lonlarini bazaga yozish
def insert_jobs(cursor, jobs):
    # executemany() — bir nechta yozuvlarni birdan bazaga kiritish imkonini beradi
    cursor.executemany(
        """
        INSERT INTO jobs (id, job_title, creator, address, date)
        VALUES (?, ?, ?, ?, ?)
        """,
        jobs,
    )


# 4️. QADAM: Bazadagi barcha ish e'lonlarini ko‘rsatish
def display_jobs(cursor):
    # Bazadan ma'lumotlarni olish uchun SELECT so‘rovini yuboramiz
    cursor.execute("""SELECT id, job_title, creator, address, date FROM jobs""")

    # Har bir yozuvni navbatma-navbat ekranga chiqaramiz
    for job in cursor.fetchall():
        print(
            f"ID: {job[0]}, Ish nomi: {job[1]}, "
            f"Yaratuvchi: {job[2]}, Manzil: {job[3]}, Sana: {job[4]}"
        )


# 5️. QADAM: Barcha jarayonlarni boshqaruvchi asosiy funksiya
def main():
    # 1-qadam: bazani tayyorlash
    conn, cursor = setup_database()

    # 2-qadam: saytdan ish e'lonlarini olish
    jobs = scrape_jobs()

    # 3-qadam: olingan ma'lumotlarni bazaga kiritish
    insert_jobs(cursor, jobs)

    # O‘zgarishlarni saqlash
    conn.commit()

    # 4-qadam: bazadagi ma'lumotlarni konsolga chiqarish
    display_jobs(cursor)

    # Oxirida bazani yopamiz
    conn.close()


#  Dastur bevosita ishga tushirilganda main() chaqiriladi
if __name__ == "__main__":
    main()


ID: 1, Ish nomi: Senior Python Developer, Yaratuvchi: Payne, Roberts and Davis, Manzil: Stewartbury, AA, Sana: 2021-04-08
ID: 2, Ish nomi: Energy engineer, Yaratuvchi: Vasquez-Davidson, Manzil: Christopherville, AA, Sana: 2021-04-08
ID: 3, Ish nomi: Legal executive, Yaratuvchi: Jackson, Chambers and Levy, Manzil: Port Ericaburgh, AA, Sana: 2021-04-08
ID: 4, Ish nomi: Fitness centre manager, Yaratuvchi: Savage-Bradley, Manzil: East Seanview, AP, Sana: 2021-04-08
ID: 5, Ish nomi: Product manager, Yaratuvchi: Ramirez Inc, Manzil: North Jamieview, AP, Sana: 2021-04-08
ID: 6, Ish nomi: Medical technical officer, Yaratuvchi: Rogers-Yates, Manzil: Davidville, AP, Sana: 2021-04-08
ID: 7, Ish nomi: Physiological scientist, Yaratuvchi: Kramer-Klein, Manzil: South Christopher, AE, Sana: 2021-04-08
ID: 8, Ish nomi: Textile designer, Yaratuvchi: Meyers-Johnson, Manzil: Port Jonathan, AE, Sana: 2021-04-08
ID: 9, Ish nomi: Television floor manager, Yaratuvchi: Hughes-Williams, Manzil: Osbornetown, AE

In [8]:
import requests
from bs4 import BeautifulSoup
import pyodbc

def scrape_jobs():
    url = "https://realpython.github.io/fake-jobs/"
    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")

    jobs = []
    job_id = 1

    for job in soup.find_all("div", class_="card-content"):
        job_title = job.find("h2", class_="title").text.strip()
        creator = job.find("h3", class_="company").text.strip()
        address = job.find("p", class_="location").text.strip()
        date = job.find("time")["datetime"][:10]  # YYYY-MM-DD

        jobs.append((job_id, job_title, creator, address, date))
        job_id += 1

    return jobs

# 1) Avval jobs ni yaratamiz
jobs = scrape_jobs()

# 2) SQL Server ga ulanamiz
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=JobsDB;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()

# 3) Jadvalga yozamiz
cursor.executemany(
    "INSERT INTO dbo.Jobs (id, job_title, creator, address, [date]) VALUES (?, ?, ?, ?, ?)",
    jobs
)
conn.commit()

# 4) Tekshirib ko‘ramiz
cursor.execute("SELECT TOP 5 * FROM dbo.Jobs ORDER BY id;")
print(cursor.fetchall())

conn.close()

[(1, 'Senior Python Developer', 'Payne, Roberts and Davis', 'Stewartbury, AA', datetime.date(2021, 4, 8)), (2, 'Energy engineer', 'Vasquez-Davidson', 'Christopherville, AA', datetime.date(2021, 4, 8)), (3, 'Legal executive', 'Jackson, Chambers and Levy', 'Port Ericaburgh, AA', datetime.date(2021, 4, 8)), (4, 'Fitness centre manager', 'Savage-Bradley', 'East Seanview, AP', datetime.date(2021, 4, 8)), (5, 'Product manager', 'Ramirez Inc', 'North Jamieview, AP', datetime.date(2021, 4, 8))]


In [ ]:
import requests
from bs4 import BeautifulSoup
import pyodbc

def scrape_jobs():
    url = "https://realpython.github.io/fake-jobs/"
    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")

    jobs = []
    job_id = 1

    for job in soup.find_all("div", class_="card-content"):
        job_title = job.find("h2", class_="title").text.strip()
        creator = job.find("h3", class_="company").text.strip()
        address = job.find("p", class_="location").text.strip()
        date = job.find("time")["datetime"][:10]  # YYYY-MM-DD

        jobs.append((job_id, job_title, creator, address, date))
        job_id += 1

    return jobs

jobs = scrape_jobs()

#  SQL Server ga ulanamiz
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=JobsDB;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()

cursor.executemany(
    "INSERT INTO dbo.Jobs (id, job_title, creator, address, [date]) VALUES (?, ?, ?, ?, ?)",
    jobs
)
conn.commit()

cursor.execute("SELECT TOP 5 * FROM dbo.Jobs ORDER BY id;")
print(cursor.fetchall())

conn.close()